# Reproduce the article's Soubré + Sefwi-Wiawso numbers (snapshot 2026-06-08)

This notebook loads the **2026-06-08 snapshot** and reproduces the quantitative claims of both companion articles:

- [`eudr-pixel-auditierbarkeit`](https://mybytes.com/research/eudr-pixel-auditierbarkeit) — Plot 2 region comparison (Hansen 2025 vintage).
- [`eudr-update-2026`](https://mybytes.com/research/eudr-update-2026) — Plot 2 vintage-drift (Hansen 2024 vs 2025).

Every number you see here comes from `data/runs/2026-06-08/area_summary.csv` and `vintage_drift.csv`, which together are the **source of truth** for every quantitative claim in the articles.

If any assert in section 3 fails, the article and the snapshot have drifted apart — that is a publishability gate.

In [10]:
import sys
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SNAPSHOT_DIR = REPO_ROOT / 'data' / 'runs' / '2026-06-08'
print(f'Snapshot: {SNAPSHOT_DIR}')

Snapshot: /Users/gwinger/PycharmProjects/soft-commodities-volatility/experiments/mybytes_content_authority/repos/eudr-risk-pipeline/data/runs/2026-06-08


## 1 · Load the published snapshot

In [11]:
area = pd.read_csv(SNAPSHOT_DIR / 'area_summary.csv')
drift = pd.read_csv(SNAPSHOT_DIR / 'vintage_drift.csv')
audit = pd.read_csv(SNAPSHOT_DIR / 'audit_trail_sample.csv')
area

,aoi_id,aoi_name,country,window_size_km,center_lon,center_lat,hansen_vintage,gfc_asset,plantation_layer_id,plantation_layer_version,plantation_threshold_tau,treecover2000_threshold_pct,hansen_lossyear_cutoff,plantation_ha,risk_ha,risk_share_pct,topology,radd_activity_level,run_timestamp
0,civ_soubre_33km,"Soubré, CIV",CIV,33,-6.6033,5.7833,hansen_v2024_v1_12,UMD/hansen/global_forest_change_2024_v1_12,projects/forestdatapartnership/assets/cocoa/mo...,2025a,0.5,30,21,73522.7,1679.4,2.28,smallholder_mosaic,low,2026-06-08T00:00:00Z
1,civ_soubre_33km,"Soubré, CIV",CIV,33,-6.6033,5.7833,hansen_v2025_v1_13,UMD/hansen/global_forest_change_2025_v1_13,projects/forestdatapartnership/assets/cocoa/mo...,2025a,0.5,30,21,73522.7,2013.3,2.74,smallholder_mosaic,low,2026-06-08T00:00:00Z
2,gha_sefwi_wiawso_33km,"Sefwi-Wiawso, GHA",GHA,33,-2.5000,6.2000,hansen_v2024_v1_12,UMD/hansen/global_forest_change_2024_v1_12,projects/forestdatapartnership/assets/cocoa/mo...,2025a,0.5,30,21,76522.9,3922.6,5.13,commercial_large_blocks,high,2026-06-08T00:00:00Z
3,gha_sefwi_wiawso_33km,"Sefwi-Wiawso, GHA",GHA,33,-2.5000,6.2000,hansen_v2025_v1_13,UMD/hansen/global_forest_change_2025_v1_13,projects/forestdatapartnership/assets/cocoa/mo...,2025a,0.5,30,21,76522.9,4633.3,6.05,commercial_large_blocks,high,2026-06-08T00:00:00Z


In [12]:
drift

,aoi_id,aoi_name,risk_share_2024_pct,risk_share_2025_pct,delta_pp,run_timestamp
0,civ_soubre_33km,"Soubré, CIV",2.28,2.74,0.45,2026-06-08T00:00:00Z
1,gha_sefwi_wiawso_33km,"Sefwi-Wiawso, GHA",5.13,6.05,0.93,2026-06-08T00:00:00Z


## 2 · Audit-trail sample (Plot 3 of `eudr-pixel-auditierbarkeit`)

In [13]:
audit

,aoi_id,lat,lon,lossyear,treecover2000_pct,gfc_tile_id,gfc_version,plantation_probability,plantation_layer_id,plantation_layer_version,threshold_tau,run_timestamp
0,civ_soubre_33km,5.6504,-6.6488,24,33,10N_010W,UMD/hansen/global_forest_change_2025_v1_13,0.9294,projects/forestdatapartnership/assets/cocoa/mo...,2025a,0.5,2026-06-08T00:00:00Z
1,civ_soubre_33km,5.6751,-6.6103,21,39,10N_010W,UMD/hansen/global_forest_change_2025_v1_13,0.9098,projects/forestdatapartnership/assets/cocoa/mo...,2025a,0.5,2026-06-08T00:00:00Z
2,civ_soubre_33km,5.6525,-6.6618,21,55,10N_010W,UMD/hansen/global_forest_change_2025_v1_13,0.9804,projects/forestdatapartnership/assets/cocoa/mo...,2025a,0.5,2026-06-08T00:00:00Z
3,civ_soubre_33km,5.7183,-6.6612,24,41,10N_010W,UMD/hansen/global_forest_change_2025_v1_13,0.9569,projects/forestdatapartnership/assets/cocoa/mo...,2025a,0.5,2026-06-08T00:00:00Z
4,civ_soubre_33km,5.7948,-6.4591,23,45,10N_010W,UMD/hansen/global_forest_change_2025_v1_13,0.9882,projects/forestdatapartnership/assets/cocoa/mo...,2025a,0.5,2026-06-08T00:00:00Z


## 3 · Publishability gate — asserts on every article number

The block below asserts that the snapshot reproduces:

- The Hansen-2025 vintage risk shares quoted in `eudr-pixel-auditierbarkeit` (Soubré **2.74 %**, Sefwi-Wiawso **6.05 %**).
- The vintage-drift quoted in `eudr-update-2026` (Soubré **2.28 → 2.74 %**, Sefwi-Wiawso **5.13 → 6.05 %**).

Tolerance is **± 0.05 percentage points** to absorb FDP-rounding noise.

In [14]:
EXPECTED_2025 = {
    'civ_soubre_33km': 2.74,
    'gha_sefwi_wiawso_33km': 6.05,
}
EXPECTED_DRIFT = {
    'civ_soubre_33km': (2.28, 2.74),
    'gha_sefwi_wiawso_33km': (5.13, 6.05),
}
TOL = 0.05

vintage_2025 = area[area['hansen_vintage'] == 'hansen_v2025_v1_13'].set_index('aoi_id')
for aoi_id, expected in EXPECTED_2025.items():
    actual = float(vintage_2025.loc[aoi_id, 'risk_share_pct'])
    assert abs(actual - expected) < TOL, (
        f'2025 vintage mismatch for {aoi_id}: snapshot {actual:.2f}% vs article {expected:.2f}%'
    )
    print(f'{aoi_id} 2025-vintage: {actual:.2f} %  ✔  matches article')

drift_idx = drift.set_index('aoi_id')
for aoi_id, (exp_24, exp_25) in EXPECTED_DRIFT.items():
    act_24 = float(drift_idx.loc[aoi_id, 'risk_share_2024_pct'])
    act_25 = float(drift_idx.loc[aoi_id, 'risk_share_2025_pct'])
    assert abs(act_24 - exp_24) < TOL, f'2024 vintage mismatch {aoi_id}: {act_24} vs {exp_24}'
    assert abs(act_25 - exp_25) < TOL, f'2025 vintage mismatch {aoi_id}: {act_25} vs {exp_25}'
    print(f'{aoi_id} drift: {act_24:.2f} → {act_25:.2f} %  ✔  matches article')

print('\nAll article numbers reproduce from the 2026-06-08 snapshot. Publishability gate green.')

civ_soubre_33km 2025-vintage: 2.74 %  ✔  matches article
gha_sefwi_wiawso_33km 2025-vintage: 6.05 %  ✔  matches article
civ_soubre_33km drift: 2.28 → 2.74 %  ✔  matches article
gha_sefwi_wiawso_33km drift: 5.13 → 6.05 %  ✔  matches article

All article numbers reproduce from the 2026-06-08 snapshot. Publishability gate green.
